### Things to do:

- Create GUI interface to easily insert the numbers and display the solved Sudoku
- Add progress bar or similar to the GUI
- Implement updated solving mechanism, as harder Sukokus are not solvable by current implementation

For the last point I have to think about how I want to go about it. Should I deal with probabilities or brute force? How should I keep track of possible solutions which where tried out already? How do I access the cells which are missing and put in one of the possible numbers?

In [1]:
# import libraries
import numpy as np
import pandas as pd
from collections import defaultdict

In [2]:
# load the sudoku
unsolved_sudoku = pd.read_csv('unsolved_sudoku_06extreme.csv',header=None)
# extract filled squares
coordinates = defaultdict(list)
for i,row in unsolved_sudoku.iterrows():
    for j,col in enumerate(row.to_numpy()):
        if np.isnan(col):
            continue
        coordinates[int(col)-1].append((i,j))

In [3]:
# initialise solver matrix
solver_matrix = np.ones((9,9,9),dtype='int8')

In [4]:
# initialise sub squares
Q11 = (slice(0,3),slice(0,3))
Q12 = (slice(0,3),slice(3,6))
Q13 = (slice(0,3),slice(6,9))
Q21 = (slice(3,6),slice(0,3))
Q22 = (slice(3,6),slice(3,6))
Q23 = (slice(3,6),slice(6,9))
Q31 = (slice(6,9),slice(0,3))
Q32 = (slice(6,9),slice(3,6))
Q33 = (slice(6,9),slice(6,9))
Q = [Q11,Q12,Q13,Q21,Q22,Q23,Q31,Q32,Q33]
def decide_Q(r,c):
    if r<3:
        if c<3:
            return Q11
        elif c>2 and c<6:
            return Q12
        else:
            return Q13
    elif r>2 and r<6:
        if c<3:
            return Q21
        elif c>2 and c<6:
            return Q22
        else:
            return Q23
    else:
        if c<3:
            return Q31
        elif c>2 and c<6:
            return Q32
        else:
            return Q33
def create_box_shapes(i,direction='horizontal'):
    t = np.zeros(3*3,dtype='bool').reshape(3,3)
    if direction=='horizontal':
        t[i,:]=True
    else:
        t[:,i]=True
    return t
box_check = {i:[create_box_shapes(i,'horizontal'),create_box_shapes(i,'vertical')] for i in range(3)}
check_vector1 = np.zeros(9,dtype='bool')
check_vector1[:3]=True
check_vector2 = np.zeros(9,dtype='bool')
check_vector2[3:6]=True
check_vector3 = np.zeros(9,dtype='bool')
check_vector3[6:]=True

In [5]:
for key,coords in coordinates.items():
    coords = np.array(coords)
    for r,c in coords:
        solver_matrix[key][decide_Q(r,c)] = 0
    solver_matrix[key][coords[:,0],:] = 0
    solver_matrix[key][:,coords[:,1]] = 0
    solver_matrix[key][coords[:,0],coords[:,1]] = 10

In [6]:
status_solved = np.sum(solver_matrix==10,axis=(1,2))

In [7]:
def cleanup_matrix(matrix):
    rows,cols = np.where(matrix.sum(axis=0)>10)
    for i in range(matrix.shape[0]):
        for r,c in zip(rows,cols):
            r,c = int(r),int(c)
            if matrix[i][r,c]!=10:
                matrix[i][r,c]=0
    return matrix
def resolve_ones(matrix,binary_mask):
    rows,cols = np.where(binary_mask)
    for i in range(matrix.shape[0]):
        for r,c in zip(rows,cols):
            r,c = int(r),int(c)
            if matrix[i][r,c] == 1:
                matrix[i][r,:] = 0
                matrix[i][:,c] = 0
                matrix[i][decide_Q(r,c)] = 0
                matrix[i][r,c]=10
    return matrix
def resolve_hidden_singles(matrix):
    for i in range(matrix.shape[0]):
        layer = matrix[i]
        for r in range(9):
            cs = np.where(layer[r,:]==1)[0]
            if cs.size == 1:
                c = int(cs[0])
                matrix[:,r,c] = 0
                matrix[i][r,:] = 0
                matrix[i][:,c] = 0
                matrix[i][decide_Q(r,c)] = 0
                matrix[i][r,c] = 10
        for c in range(9):
            rs = np.where(layer[:,c]==1)[0]
            if rs.size == 1:
                r = int(rs[0])
                matrix[:,r,c] = 0
                matrix[i][r,:] = 0
                matrix[i][:,c] = 0
                matrix[i][decide_Q(r,c)] = 0
                matrix[i][r,c] = 10
        for q in Q:
            rr,cc = np.where(layer[q]==1)
            if rr.size == 1:
                r,c = q[0].start+int(rr[0]), q[1].start+int(cc[0])
                matrix[:,r,c] = 0
                matrix[i][r,:] = 0
                matrix[i][:,c] = 0
                matrix[i][decide_Q(r,c)] = 0
                matrix[i][r,c] = 10
    return matrix
def resolving_pointing_numbers(matrix):
    for i in range(matrix.shape[0]):
        layer = matrix[i]
        for q in Q:
            box_mask = layer[q]==1
            box_mask_sum = np.sum(box_mask)
            if box_mask_sum in (2,3):
                for rc,masks in box_check.items():
                    for j,mask in enumerate(masks):
                        overlap = np.sum(box_mask & mask)
                        if (box_mask_sum==2 and overlap==2) or (box_mask_sum==3 and overlap==3):
                            coords = np.array(np.where(box_mask)).T+np.array([q[0].start,q[1].start])
                            if j==0:
                                matrix[i][rc+q[0].start,:] = 0
                            else:
                                matrix[i][:,rc+q[1].start] = 0
                            matrix[i][coords[:,0],coords[:,1]] = 1
    return matrix
check_vector1 = np.zeros(9,dtype='bool')
check_vector1[:3]=True
check_vector2 = np.zeros(9,dtype='bool')
check_vector2[3:6]=True
check_vector3 = np.zeros(9,dtype='bool')
check_vector3[6:]=True
def clean_box(matrix,index,Q,fill_boolean,row=True):
    matrix[Q]=0
    if row:
        matrix[index,fill_boolean]=1
    else:
        matrix[fill_boolean,index]=1
    return matrix
def resolving_claiming_boxes(matrix):
    for i in range(matrix.shape[0]):
        layer = matrix[i]
        for r in range(9):
            row_ones = layer[r,:]==1
            row_ones_sum = row_ones.sum()
            if row_ones_sum in [2,3]:
                if (np.sum(row_ones & check_vector1)==2 and row_ones_sum==2) or (np.sum(row_ones & check_vector1)==3 and row_ones_sum==3):
                    layer = clean_box(layer,r,decide_Q(r,0),row_ones & check_vector1)
                elif (np.sum(row_ones & check_vector2)==2 and row_ones_sum==2) or (np.sum(row_ones & check_vector2)==3 and row_ones_sum==3):
                    layer = clean_box(layer,r,decide_Q(r,3),row_ones & check_vector2)
                elif (np.sum(row_ones & check_vector3)==2 and row_ones_sum==2) or (np.sum(row_ones & check_vector3)==3 and row_ones_sum==3):
                    layer = clean_box(layer,r,decide_Q(r,6),row_ones & check_vector3)
        for c in range(9):
            col_ones = layer[:,c]==1
            col_ones_sum = col_ones.sum()
            if col_ones_sum in [2,3]:
                if (np.sum(col_ones & check_vector1)==2 and col_ones_sum==2) or (np.sum(col_ones & check_vector1)==3 and col_ones_sum==3):
                    layer = clean_box(layer,c,decide_Q(0,c),col_ones & check_vector1,row=False)
                elif (np.sum(col_ones & check_vector2)==2 and col_ones_sum==2) or (np.sum(col_ones & check_vector2)==3 and col_ones_sum==3):
                    layer = clean_box(layer,c,decide_Q(3,c),col_ones & check_vector2,row=False)
                elif (np.sum(col_ones & check_vector3)==2 and col_ones_sum==2) or (np.sum(col_ones & check_vector3)==3 and col_ones_sum==3):
                    layer = clean_box(layer,c,decide_Q(6,c),col_ones & check_vector3,row=False)
    return matrix 
def is_sudoku_solved(matrix):
    solved_sudoku = np.argmax(matrix,axis=0)+1
    sub_square_check = np.all([np.unique(solved_sudoku[q]).shape[0]==9 for q in Q])
    vertical_check = np.all([np.unique(solved_sudoku[i,:]).shape[0]==9 for i in range(solved_sudoku.shape[0])])
    horizontal_check = np.all([np.unique(solved_sudoku[:,i]).shape[0]==9 for i in range(solved_sudoku.shape[0])])
    if np.all([sub_square_check,vertical_check,horizontal_check]):
        print('Sudoku solved')
        return True
    else:
        return False

In [8]:
def solving_loop(matrix):
    continue_loop = True
    matrix = cleanup_matrix(matrix)
    remaining_ones = np.sum(matrix==1)
    while continue_loop:
        z_sum = np.sum(matrix,axis=0)
        z_sum_binary = z_sum == 1
        if np.any(z_sum_binary):
            matrix = resolve_ones(matrix,z_sum_binary)
        else:
            matrix = resolve_hidden_singles(matrix)
            matrix = resolving_pointing_numbers(matrix)
            matrix = resolving_claiming_boxes(matrix)
        decision_check = np.sum(matrix==1)
        if np.any(matrix.sum(axis=0)==0):
            status = 'dead end'
            return 'dead end',matrix
        if decision_check==remaining_ones:
            return 'decision',matrix
        if np.all(matrix.sum(axis=0)==10):
            return 'solved',matrix
        remaining_ones = decision_check

In [9]:
%%time
status,matrix = solving_loop(solver_matrix)
print(status)

solved
CPU times: user 19.5 ms, sys: 4.99 ms, total: 24.5 ms
Wall time: 20.5 ms


In [10]:
def recursive_solve(status,matrix,depth):
    rows,cols = np.where(matrix.sum(axis=0)==2)
    if rows.size==0:
        rows,cols = np.where(matrix.sum(axis=0)>2)
    for r,c in zip(rows,cols):
        z_index = np.where(matrix[:,r,c]==1)[0]
        for z in z_index:
            new_matrix = matrix.copy()
            new_matrix[:,r,c] = 0
            new_matrix[z][r,:] = 0
            new_matrix[z][:,c] = 0
            new_matrix[z][decide_Q(r,c)] = 0
            new_matrix[z][r,c] = 10
            new_status,new_matrix = solving_loop(new_matrix)
            if new_status == 'solved':
                return new_status,new_matrix,depth
            elif new_status == 'decision':
                results_status,results_matrix = recursive_solve(new_status,new_matrix)
                if results_status == 'solved':
                    return results_status,results_matrix
            else:
                continue
    return 'dead end', matrix,depth

In [11]:
# %%time
status,solver_matrix,depth = recursive_solve(status,matrix,0)

In [35]:
np.argmax(solver_matrix,axis=0)+1

array([[7, 3, 8, 2, 5, 9, 6, 4, 1],
       [2, 9, 1, 4, 7, 6, 3, 8, 5],
       [6, 5, 4, 3, 8, 1, 7, 9, 2],
       [3, 8, 7, 6, 4, 2, 1, 5, 9],
       [4, 1, 5, 7, 9, 8, 2, 6, 3],
       [9, 6, 2, 5, 1, 3, 8, 7, 4],
       [1, 4, 9, 8, 2, 7, 5, 3, 6],
       [5, 7, 6, 1, 3, 4, 9, 2, 8],
       [8, 2, 3, 9, 6, 5, 4, 1, 7]])

In [20]:
is_sudoku_solved(solver_matrix),is_sudoku_solved(matrix)

Sudoku solved
Sudoku solved


(True, True)

In [34]:
depth

0